<a href="https://colab.research.google.com/github/shivansh2310/The-elements-of-quantitative-investing-/blob/main/Fundamental_Factor_Models_(Chapter_6).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###A. The Cross-Sectional Equation

The math flips.For a single time period $t$, the return vector of $N$ assets ($R$) is defined as:

$$R = X \cdot f + u$$

* $X$ (The Exposure Matrix): An $N \times K$ matrix. These are the known characteristics of your assets (e.g., Size, Value, Momentum, and Sector dummy variables).
* $f$ (The Factor Returns): A $K \times 1$ vector. This is what you are solving for. It tells you exactly how much the market rewarded or punished a specific characteristic on that day.

* $u$ (The Specific Returns): An $N \times 1$ vector. The true idiosyncratic, orthogonal alpha left over after stripping out the fundamental factors.

###B. Z-Scoring (Standardization)

You cannot run a regression with Market Cap (in billions of dollars) and Dividend Yield (in percentages) in the same matrix. The condition number will explode. Every continuous characteristic must be cross-sectionally standardized (Z-scored) so that they have a mean of 0 and a standard deviation of 1.

$$Z_{i} = \frac{x_{i} - \mu_{x}}{\sigma_{x}}$$

###C. Dummy Variables for Sectors

Sectors aren't continuous; they are categorical. You use a binary matrix (1 if in the sector, 0 if not). Crucial Trap: If you include the market factor (an intercept of 1s) AND dummy variables for every sector, your matrix will be rank-deficient (perfect multicollinearity) and cannot be inverted. You must either drop the intercept (market factor) or drop one sector to act as the baseline.

## Engineering the Cross-Sectional Engine

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import yfinance as yf
import statsmodels.api as sm

In [26]:
stock_data = {
    "MSFT": "Tech", "NVDA": "Tech", "AAPL": "Tech",
    "NEE": "Utilities", "AEP": "Utilities", "VST": "Utilities",
    "CAT": "Industrials", "GE": "Industrials", "PWR": "Industrials",
    "JPM": "Financials", "GS": "Financials", "V": "Financials",
    "XOM": "Energy", "CVX": "Energy",
    "META": "Comm", "GOOGL": "Comm",
    "LLY": "Healthcare", "UNH": "Healthcare",
    "AMZN": "Consumer", "WMT": "Consumer"
}

tickers = list(stock_data.keys())
data = []

print("Fetching structural data from Yahoo Finance... (This takes 10-20 seconds)")

# 2. Data Acquisition
for ticker in tickers:
    try:
        tk = yf.Ticker(ticker)
        info = tk.info

        # Get Fundamental Characteristics
        mcap = info.get('marketCap', np.nan)
        pe = info.get('trailingPE', np.nan)

        # Get simple daily return (using 5d history to ensure we catch the latest close)
        hist = tk.history(period="5d")
        if len(hist) >= 2:
            latest_return = hist['Close'].pct_change().iloc[-1]
        else:
            latest_return = np.nan

        data.append({
            "Ticker": ticker,
            "Sector": stock_data[ticker],
            "Return": latest_return,
            "MarketCap": mcap,
            "PE": pe
        })
    except Exception as e:
        print(f"Failed to fetch {ticker}")

# Create DataFrame and drop missing values
df = pd.DataFrame(data).set_index("Ticker").dropna()

Fetching structural data from Yahoo Finance... (This takes 10-20 seconds)


In [27]:
df['EarningsYield'] = 1 / df['PE']

# Z-Score the Exposures (Crucial to prevent matrix explosion)
df['Size_Z'] = (df['MarketCap'] - df['MarketCap'].mean()) / df['MarketCap'].std()
df['Value_Z'] = (df['EarningsYield'] - df['EarningsYield'].mean()) / df['EarningsYield'].std()

# Categorical Dummy Variables
# drop_first=True prevents the "Dummy Variable Trap" (Perfect Multicollinearity)
# If we keep all sectors AND an intercept, the matrix becomes rank-deficient.
dummies = pd.get_dummies(df['Sector'], prefix='Sector', drop_first=True, dtype=int)
df = pd.concat([df, dummies], axis=1)

In [28]:
df

,Sector,Return,MarketCap,PE,EarningsYield,Size_Z,Value_Z,Sector_Consumer,Sector_Energy,Sector_Financials,Sector_Healthcare,Sector_Industrials,Sector_Tech,Sector_Utilities
Ticker,,,,,,,,,,,,,,
MSFT,Tech,-0.026586,3095206035456,24.831347,0.040272,1.056536,0.349699,0,0,0,0,0,1,0
NVDA,Tech,-0.062014,4967727366144,31.457056,0.031789,2.205187,-0.308756,0,0,0,0,0,1,0
AAPL,Tech,-0.012499,4514011676672,37.163240,0.026908,1.926866,-0.687657,0,0,0,0,0,1,0
NEE,Utilities,0.009206,179028361216,21.786800,0.045899,-0.732318,0.786557,0,0,0,0,0,0,1
AEP,Utilities,0.010564,70265724928,19.131851,0.052269,-0.799036,1.281002,0,0,0,0,0,0,1
VST,Utilities,-0.032141,50159259648,24.876253,0.040199,-0.811369,0.344055,0,0,0,0,0,0,1
CAT,Industrials,-0.038491,416504119296,45.033867,0.022206,-0.586645,-1.052721,0,0,0,0,1,0,0
GE,Industrials,0.001068,342704128000,40.796020,0.024512,-0.631915,-0.873660,0,0,0,0,1,0,0
PWR,Industrials,-0.033455,104308301824,95.220540,0.010502,-0.778153,-1.961232,0,0,0,0,1,0,0


In [29]:
Y = df['Return']

# Exposure Matrix (X)
features = ['Size_Z', 'Value_Z'] + list(dummies.columns)
X = df[features]

# Add Intercept (This acts as the baseline Market Factor)
X = sm.add_constant(X)

# 5. Fit the Model
model = sm.OLS(Y, X).fit()

# Print Results
print("\n" + "="*60)
print(" CROSS-SECTIONAL FUNDAMENTAL FACTOR MODEL RESULTS")
print("="*60)
print(model.summary())


 CROSS-SECTIONAL FUNDAMENTAL FACTOR MODEL RESULTS
                            OLS Regression Results                            
Dep. Variable:                 Return   R-squared:                       0.317
Model:                            OLS   Adj. R-squared:                 -0.298
Method:                 Least Squares   F-statistic:                    0.5149
Date:                Sat, 06 Jun 2026   Prob (F-statistic):              0.834
Time:                        11:09:52   Log-Likelihood:                 50.706
No. Observations:                  20   AIC:                            -81.41
Df Residuals:                      10   BIC:                            -71.45
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

Look directly at the coef column in the middle of the summary table. This is our $f$ vector (the factor premiums).
* const: This is the baseline market return for the day.
* Size_Z: If this coefficient is negative, it means the market heavily penalized large-cap stocks that day (a "Small Firm" premium was active). If it's positive, mega-caps outperformed.
* Value_Z: If this is positive, the market rewarded companies with high earnings yields (Value outperformed Growth).
* Sector_Tech (etc.): These show the pure sector performance independent of the Size and Value effects of the companies within them.